# Assignment 1: Do personas annotate like people, and does context change them? (starter notebook)

Read `assignment1_student.md` first. The cells marked **TODO** are yours. The other cells are given. Keep the check cells: they fail until your variable exists and has the right shape. Before you hand in, run the notebook from top to bottom.

In [ ]:
! pip install syndisco krippendorff

## Setup

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = None  # TODO: choose a seed, write it down, and use it for every random choice below

## 1. Load and clean

**TODO 1.** Load the file of 20 conversations into `items`: a dict (or a Series) from item id to the text of the conversation, with who said what and in which order. Clean only what needs it, and say what you changed below.

In [ ]:
items = None  # TODO: 20 items, item id -> clean text

In [ ]:
vals = list(items.values()) if isinstance(items, dict) else list(items)
assert len(vals) == 20, "items must hold 20 conversations"
assert all(isinstance(t, str) and t.strip() for t in vals), "every item must be a non-empty string"
assert len(set(vals)) == 20, "the items must be different"
print("ok")

What I cleaned, and why:

- TODO

## 2. Annotate by hand

**TODO 2.** Write your guidelines in the cell below **first**. Then export the items, label each one from 1 (fully civil) to 5 (very toxic) in a spreadsheet, alone and **before you look at any model's labels**, and save `human_annotations.csv` (item id, label, optional note). Load it into `human` (a dict or a Series, item id to label).

**My guidelines** (what makes a 1, a 3, a 5? what about sarcasm, or an insult aimed at an idea instead of a person?):

TODO

In [ ]:
# TODO: export the items so that you can annotate them

In [ ]:
human = None  # TODO: your labels, loaded from human_annotations.csv

In [ ]:
labels = dict(human) if isinstance(human, dict) else human.to_dict()
assert set(labels) == set(items), "one label per item"
assert all(1 <= int(v) <= 5 for v in labels.values()), "labels must be between 1 and 5"
print("ok")

## 3. Generate one conversation per item

First the model and the SynDisco classes (given). The model answers up to `max_out_tokens` tokens per call. Generating a comment needs more than labelling does, so you will have to deal with that.

In [1]:
import syndisco
from syndisco import (
    Actor,
    AnnotationExperiment,
    DiscussionExperiment,
    Logs,
    QueueTurnManager,
    RandomTurnManager,
    RespondTurnManager,
    TransformersModel,
)

model = TransformersModel(
    model_path="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=120,  # long enough for a comment; a label needs far less
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True,
)

c:\Users\swkra\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\swkra\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.05GB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\swkra\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\swkra\.cache\huggingface\hub\models--unsloth--Qwen2.5-3B-Instruct-bnb-4bit. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

**TODO 3a. Participants.** Draw random personas for the participants of the conversations (a persona is a small dictionary), using `SEED`. Keep them in `participants`.

**TODO 3b. Conversations.** For each item, run one small discussion between some of the participants, with the item as their starting context. Keep the result in `generated` (a dict: item id to the conversation as text, with who said what). **Generate one or two and read them before you do all 20.**

*Hints:* what do all the participants of a discussion share? Each item has its own context, so what does that mean for the experiments? If the conversations look wrong (cut off, refusals, repeated sentences, a text about the item instead of a reaction to it), change the output limit and the instructions.

In [ ]:
participants = None  # TODO: random personas for the participants

In [ ]:
generated = None  # TODO: item id -> the conversation generated for that item

In [ ]:
gen = dict(generated)
assert set(gen) == set(items), "one generated conversation per item"
assert all(isinstance(t, str) and t.strip() for t in gen.values()), "every conversation must be a non-empty string"
print("ok")

What the first conversations looked like, and what I changed:

- TODO

## 4. Model annotators, with and without context

Given: the baseline grid.

In [ ]:
BASELINE_GRID = {
    "gender": ["female", "male"],
    "race": ["Black", "White", "Asian", "Hispanic"],
    "politics": ["left", "center", "right"],
}

**TODO 4. Personas.** Build the personas of `BASELINE_GRID` in code (every combination of values), plus one baseline with no identity. Keep them in `personas`, a list of dictionaries (25 of them).

In [ ]:
personas = None  # TODO

In [ ]:
assert personas is not None and len(personas) == 25, "24 identities and 1 baseline"
assert len({json.dumps(p, sort_keys=True) for p in personas}) == 25, "all 25 personas must be different"
print("ok")

**TODO 5. Instructions.** Write `ANN_CONTEXT` and `ANN_INSTRUCTIONS` **once**: what does each number mean, and how must the model answer? They are identical for every annotator and for both conditions.

*Hint:* reread the annotation guide: how is an annotator-agent created, and how does it differ from a user-agent?

In [ ]:
ANN_CONTEXT = None       # TODO
ANN_INSTRUCTIONS = None  # TODO

**TODO 6. No context.** Run all 25 annotators on all 20 items, each annotator seeing only the item. **Test first** on 2 annotators and 3 items and read the output files. Then run everything.

Read the output files into `ann_no_context`, with one row per annotator and item: the identity attributes (or "baseline"), `item_id`, `raw_answer`, `label` and `valid`. Count the invalid answers per persona.

*Hints:* the output limit; how much context the annotator sees by default; what the `name` of a log entry does in the prompt; open one output file before you write the code that reads them.

In [ ]:
# TODO: small test run, then the full run (no context)

In [ ]:
ann_no_context = None  # TODO: one row per annotator and item

In [ ]:
assert len(ann_no_context) == 25 * 20, "25 annotators x 20 items"
assert {"item_id", "raw_answer", "label", "valid"} <= set(ann_no_context.columns), "missing columns"
assert set(ann_no_context["item_id"]) == set(items), "all items must be there"
print("ok")

**TODO 7. With context.** The same 25 annotators, the same instructions, the same items. The only change: each annotator first reads the conversation generated for that item, then labels **the item** (not the conversation). Keep the result in `ann_context`, in the same format.

*Hint:* where does the context go, so that the comparison with the no-context run stays fair?

In [ ]:
# TODO: small test run, then the full run (with context)

In [ ]:
ann_context = None  # TODO: one row per annotator and item

In [ ]:
assert len(ann_context) == 25 * 20, "25 annotators x 20 items"
assert {"item_id", "raw_answer", "label", "valid"} <= set(ann_context.columns), "missing columns"
assert set(ann_context["item_id"]) == set(items), "all items must be there"
print("ok")

## 5. Agreement

**TODO 8.** Compute everything in two ways: **categorical** (1-2 civil against 3-5 toxic, Cohen's kappa) and **numerical** (1-5, weighted kappa or Spearman). Compare, always on the same items (join on `item_id`):

1. you against each identity, no context;
2. you against each identity, with context;
3. each identity without against with context;
4. persona against persona within each condition: a heatmap and one overall score (which dimension separates the personas most?).

Save everything you report in `agreement.json`.

In [ ]:
# TODO: 1. you vs each identity, no context

In [ ]:
# TODO: 2. you vs each identity, with context

In [ ]:
# TODO: 3. each identity without vs with context

In [ ]:
# TODO: 4. persona vs persona: heatmap and overall score

In [ ]:
# TODO: write agreement.json

## 6. The story

**TODO 9. Notes for the report** (the report is a separate 1-2 page PDF): does context move the labels, and in which direction? Closer to you or further? Do the personas agree more or less once they have read the conversation? Which identities move most? The limitations.

- TODO

**Before you hand in:** the clean items, `human_annotations.csv` and your guidelines, `generated.jsonl`, the two annotation files, `agreement.json`, the figures, the report, and this notebook run from top to bottom.